In [ ]:
# replace the last cell of main project with this to verify the result and to use prerecorded datasets
if __name__ == "__main__":
    from scipy.io import wavfile # Add this to imports at the top if you want, or leave here
    
    # --- CHANGE THIS PATH TO YOUR WAV FILE ---
    FILE_PATH ='Sounds/367215__daalvinz__lalala.wav'
    
    print("Loading audio...")
    
    # BULLETPROOF AUDIO LOADING
    fs, audio_raw = wavfile.read(FILE_PATH)
    # fs,audio_raw = fs_test,perfect_audio
    
    # Force Mono (YIN only works on single-channel audio)
    if len(audio_raw.shape) == 2:
        audio = audio_raw[:, 0]
    else:
        audio = audio_raw
        
    # Normalize to -1.0 to 1.0 (Crucial for the math to work)
    audio = audio.astype(np.float32)
    audio = audio / np.max(np.abs(audio))
    
    # Run the YIN Algorithm
    print("Running YIN Pitch Detection...")
    pitch_contour = pitchDetect(audio, fs, min_f0=50, max_f0=1000, W=2048, decimation_factor=1, cmndf_threshold=0.30)
    
    # --- PLOTTING ---
    print("Generating plot...")
    step = 2048 // 2 # Must match the W//2 step in pitchDetect
    time_steps = np.arange(len(pitch_contour)) * step / fs
    
    plt.figure(figsize=(14, 6))
    
    # Plot the waveform (Scipy version, no Librosa needed)
    plt.subplot(2, 1, 1)
    time_audio = np.linspace(0, len(audio) / fs, len(audio))
    plt.plot(time_audio, audio, color='b', linewidth=0.5)
    plt.title("Audio Waveform")
    plt.ylabel("Amplitude")
    
    # Plot the pitch contour
    plt.subplot(2, 1, 2)
    clean_pitch = np.where(pitch_contour == None, np.nan, pitch_contour)
    plt.plot(time_steps, clean_pitch, color='r', linewidth=2)
    # plt.xlim(0, len(audio) / fs)
    plt.ylim(0, 500) 
    plt.title("Detected Fundamental Frequency (F0)")
    plt.xlabel("Time (Seconds)")
    plt.ylabel("Frequency (Hz)")
    
    plt.tight_layout()
    plt.show()
    